# Curvas de treino

Este notebook só lê o que os scripts já gravaram em `outputs/`: o `historico.json` de cada run (passos 2 e 3) e o `avaliacao.json` do `scripts/avaliar.py`. Nada é treinado aqui.

Para ter o que plotar, rode antes os scripts com `--saida` diferente para cada experimento. Os comandos estão no README.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))

from plotar_curvas import carregar_curvas, plotar, runs_disponiveis

avaliacao = json.loads((RAIZ / "outputs" / "avaliacao.json").read_text())
PERDA_BASE = avaliacao["base"]["eval_loss"]   # o modelo sem treino nenhum
runs = runs_disponiveis()
print(f"perda de validação do modelo base: {PERDA_BASE:.4f}")
print("runs encontrados:", ", ".join(runs))

## Overfitting

Mesmo modelo, mesmo learning rate, 3 épocas contra 15. As linhas tracejadas (treino) descem sem parar; a linha cheia (validação) faz um "U". O fundo do "U" é o melhor modelo, e é ele que o `load_best_model_at_end` restaura.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plotar([r for r in ["sft-completo", "sft-ep15"] if r in runs], ax=ax, base=PERDA_BASE)
ax.set_title("Fine-tuning completo: 3 contra 15 épocas");

## Learning rate alto demais

`lr=5e-4` contra o padrão `1e-4`. A curva de validação é só metade da história: a coluna `ppl geral` da tabela no fim do notebook mostra quanto do pré-treino cada run apagou.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plotar([r for r in ["sft-completo", "sft-lr5e-4"] if r in runs], ax=ax, base=PERDA_BASE)
ax.set_title("Fine-tuning completo: lr 1e-4 contra 5e-4");

## LoRA: rank e alvos

Rank 8 em `q_proj`/`v_proj` (padrão), rank 32 nos mesmos alvos, e rank 8 em todas as projeções lineares. O fine-tuning completo entra como referência.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plotar([r for r in ["sft-completo", "lora", "lora-r32", "lora-todos"] if r in runs], ax=ax, base=PERDA_BASE)
ax.set_title("LoRA contra fine-tuning completo");

## Mais dados

O dataset grande soma 38 termos novos ao treino, e a validação continua **idêntica**. A única variável é a quantidade de dados de treino.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plotar([r for r in ["sft-completo", "sft-grande"] if r in runs], ax=ax, base=PERDA_BASE)
ax.set_title("38 contra 76 termos no treino (mesma validação)");

## Perda contra qualidade de geração

A tabela do `avaliar.py`. Compare a ordem da coluna `val loss` com a das colunas `formato` e `parou`: o modelo de menor perda não é necessariamente o que gera melhor.

In [ ]:
print(f"{'modelo':<16} {'val loss':>8} {'formato':>8} {'parou':>6} {'tokens':>6} {'F1':>5} {'ppl geral':>9}")
for nome, r in avaliacao.items():
    if "formato" not in r:
        continue
    print(f"{nome:<16} {r['eval_loss']:>8.4f} {r['formato']:>8.0%} {r['parou']:>6.0%} "
          f"{r['tokens']:>6.0f} {r['f1']:>5.2f} {r['ppl_geral']:>9.2f}")

In [ ]:
nomes = [n for n, r in avaliacao.items() if "formato" in r]
fig, ax = plt.subplots(figsize=(7, 5))
for nome in nomes:
    r = avaliacao[nome]
    ax.scatter(r["eval_loss"], r["formato"], s=60)
    ax.annotate(nome, (r["eval_loss"], r["formato"]), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("melhor perda de validação")
ax.set_ylabel("respostas com o formato correto")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.grid(alpha=0.3)
ax.set_title("Perda de validação não é qualidade de geração");